In [ ]:
!pip uninstall -y bitsandbytes triton -q 2>/dev/null
!pip install -q transformers==4.46.3 trl==0.12.0 peft==0.14.0 accelerate==1.2.1 datasets pandas lxml

In [ ]:
import os
import re
import time
import random
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch
from datasets import Dataset

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
MODEL_SAVE_PATH = "/content/drive/MyDrive/qwen15b_svg_lora_v9"
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
print(f"Model will be saved to: {MODEL_SAVE_PATH}")

In [ ]:
# load dataset
TRAIN_PATH = "/content/train.csv"
TEST_PATH = "/content/test.csv"

train_df = pd.read_csv(TRAIN_PATH, engine="python")
test_df = pd.read_csv(TEST_PATH, engine="python")
print(f"Original training samples: {len(train_df)}")
print(f"Columns: {train_df.columns.tolist()}")

In [ ]:
# filter to keep only SVGs with simple geometric shapes.
# this may improve model learning for specifically test set requirements
def is_simple_svg(svg_text):
  svg_text = str(svg_text)

  # basic validation
  if len(svg_text) < 50 or len(svg_text) > 5000:
      return False
  if not svg_text.startswith('<svg'):
      return False

  # color is required
  has_color = 'fill=' in svg_text or 'stroke=' in svg_text
  if not has_color:
      return False

  # simple shapes prefix prefered
  has_circle = '<circle' in svg_text
  has_rect = '<rect' in svg_text
  has_polygon = '<polygon' in svg_text
  has_path = '<path' in svg_text

  if has_circle or has_rect or has_polygon:
      return True

  if has_path:
      import re
      path_match = re.search(r'<path[^>]*d="([^"]*)"', svg_text)
      if path_match and len(path_match.group(1)) < 60:
          return True

  return False

train_df['is_simple'] = train_df['svg'].apply(is_simple_svg)
simple_df = train_df[train_df['is_simple']].copy()
print(f"Simple shape samples: {len(simple_df)} / {len(train_df)}")

In [ ]:
# sample to control training time
SAMPLE_SIZE = min(4000, len(simple_df))
if len(simple_df) > SAMPLE_SIZE:
  simple_df = simple_df.sample(n=SAMPLE_SIZE, random_state=SEED)
  print(f"Sampled to: {len(simple_df)} samples")
else:
  print(f"Using all {len(simple_df)} samples")

train_ds = Dataset.from_pandas(simple_df[['prompt', 'svg']])
splits = train_ds.train_test_split(test_size=0.05, seed=SEED)
train_ds = splits['train']
eval_ds = splits['test']

print(f"Training samples: {len(train_ds)}")
print(f"Validation samples: {len(eval_ds)}")

In [ ]:
# check if SVGs are valid
valid_count = 0
invalid_count = 0
for i in range(min(100, len(train_ds))):
  svg = train_ds[i]['svg']
  try:
    root = ET.fromstring(svg)
    if root.tag.endswith('svg'):
      valid_count += 1
    else:
      invalid_count += 1
  except:
    invalid_count += 1

print(f"Valid SVGs: {valid_count}/100")
print(f"Invalid SVGs: {invalid_count}/100")

In [ ]:
SYSTEM_PROMPT = (
    "You generate compact, valid SVG markup from user requests. "
    "Return only SVG code with a single root <svg> element."
)

def format_sft_text(example):
    text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{example['prompt']}<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{example['svg']}<|im_end|>"
    )
    return {"text": text}

train_text = train_ds.map(format_sft_text, remove_columns=train_ds.column_names)
eval_text = eval_ds.map(format_sft_text, remove_columns=eval_ds.column_names)

print(train_text[0]["text"][:400])

In [ ]:
CONFIG = {
    "model_name": "Qwen/Qwen2.5-1.5B-Instruct",
    "max_seq_length": 512,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.1,
    "learning_rate": 2e-4,
    "num_train_epochs": 2,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "warmup_steps": 100,
    "weight_decay": 0.01,
    "logging_steps": 20,
    "eval_steps": 100,
    "save_steps": 200,
    "output_dir": MODEL_SAVE_PATH,
}

for k, v in CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"], trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

lora_config = LoraConfig(
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTrainable: {trainable_params:,} / {total_params:,} ({100 * trainable_params / total_params:.2f}%)")
if torch.cuda.is_available():
    print(f"GPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=CONFIG["max_seq_length"],
    )

tokenized_train = train_text.map(tokenize_function, batched=True)
tokenized_eval = eval_text.map(tokenize_function, batched=True)

# remove original text column to save memory
tokenized_train = tokenized_train.remove_columns(["text"])
tokenized_eval = tokenized_eval.remove_columns(["text"])

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # not masked
)

# training args
training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_steps=CONFIG["warmup_steps"],
    weight_decay=CONFIG["weight_decay"],
    fp16=True,
    logging_steps=CONFIG["logging_steps"],
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    seed=SEED,
    remove_unused_columns=False,
)

trainer = Trainer(
  model=model,
  args=training_args,
  train_dataset=tokenized_train,
  eval_dataset=tokenized_eval,
  data_collator=data_collator,
)

print("\nStarting training...")
train_start = time.time()
train_result = trainer.train()
train_time = (time.time() - train_start) / 60
print(f"\nTraining completed! Time: {train_time:.2f} minutes")

os.makedirs(CONFIG["output_dir"], exist_ok=True)
model.save_pretrained(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])
print(f"Model saved to: {CONFIG['output_dir']}")

In [ ]:
model.eval()

# Extract SVG code from generated text.
# Handle both complete and truncated SVG tags.
def extract_svg(text):
    pattern = r'<svg\s[^>]*xmlns[^>]*>.*?</svg>'
    match = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(0).strip()

    # fallback
    pattern2 = r'<svg[^>]*>.*?</svg>'
    match = re.search(pattern2, text, re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(0).strip()

    return ""

def fallback_svg():
    return (
        '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
        '<rect width="256" height="256" fill="white"/>'
        '<circle cx="128" cy="128" r="64" fill="black"/>'
        '</svg>'
    )

def generate_svg(prompt, max_new_tokens=384):
    chat_text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{prompt}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )
    inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    svg = extract_svg(decoded)

    # validate extracted svg
    is_valid = False
    if svg:
        try:
            root = ET.fromstring(svg)
            is_valid = root.tag.endswith("svg")
        except:
            pass

    return svg, is_valid

# test generation
svg, is_valid = generate_svg("a simple red circle")
print(f"Test generation valid: {is_valid}")

In [ ]:
test_prompts = test_df['prompt'].tolist()
print(f"Generating {len(test_prompts)} SVGs...")

t0 = time.time()
svgs = []
valid_count = 0

for idx, prompt in enumerate(test_prompts):
    svg, is_valid = generate_svg(prompt)
    if not svg or not is_valid:
        svg = fallback_svg()
    else:
        valid_count += 1
    svgs.append(svg)

    if (idx + 1) % 100 == 0:
        print(f"  {idx + 1}/{len(test_prompts)} | Valid: {valid_count}/{idx+1}")

print(f"\nValid SVGs: {valid_count}/{len(test_df)}")

rows = [{"id": test_df.iloc[i]["id"], "svg": svgs[i]} for i in range(len(test_df))]
sub_df = pd.DataFrame(rows)

# verify no null values in submission
print(f"Null values: {sub_df['svg'].isnull().sum()}")

sub_df.to_csv("/content/submission.csv", index=False)
print("Submission saved: /content/submission.csv")

from google.colab import files
files.download("/content/submission.csv")